Scratch from [Machine learning mastery](https://machinelearningmastery.com/building-transformer-models-from-scratch-with-pytorch-10-day-mini-course/)

# Lesson 1: Getting the Data

Can you think about why having diverse genres is important when building a language model?  
Diverse genres expose a language model to different vocabulary, writing styles, sentence structures, and topics. This helps the model learn broader language patterns, generalize to unfamiliar text, and handle different tasks more effectively. It also reduces the risk of overfitting to a particular genre or writing style.

In [1]:
import os
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import tokenizers
import tqdm

In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Download novels from Project Gutenberg
DATASOURCE = {
    "memoirs_of_grant": "https://www.gutenberg.org/ebooks/4367.txt.utf-8",
    "frankenstein": "https://www.gutenberg.org/ebooks/84.txt.utf-8",
    "sleepy_hollow": "https://www.gutenberg.org/ebooks/41.txt.utf-8",
    "origin_of_species": "https://www.gutenberg.org/ebooks/2009.txt.utf-8",
    "makers_of_many_things": "https://www.gutenberg.org/ebooks/28569.txt.utf-8",
    "common_sense": "https://www.gutenberg.org/ebooks/147.txt.utf-8",
    "economic_peace": "https://www.gutenberg.org/ebooks/15776.txt.utf-8",
    "the_great_war_3": "https://www.gutenberg.org/ebooks/29265.txt.utf-8",
    "elements_of_style": "https://www.gutenberg.org/ebooks/37134.txt.utf-8",
    "problem_of_philosophy": "https://www.gutenberg.org/ebooks/5827.txt.utf-8",
    "nights_in_london": "https://www.gutenberg.org/ebooks/23605.txt.utf-8",
}
for filename, url in DATASOURCE.items():
    if not os.path.exists(f"{filename}.txt"):
        response = requests.get(url)
        with open(f"{filename}.txt", "wb") as f:
            f.write(response.content)

# Read and preprocess the text
def preprocess_gutenberg(filename):
    with open(filename, "r", encoding="utf-8") as f:
        text = f.read()

    # Find the start and end of the actual content
    start = text.find("*** START OF THE PROJECT GUTENBERG EBOOK")
    start = text.find("\n", start) + 1
    end = text.find("*** END OF THE PROJECT GUTENBERG EBOOK")

    # Extract the main content
    text = text[start:end].strip()

    # Basic preprocessing
    # Remove multiple newlines and spaces
    text = "\n".join(line.strip() for line in text.split("\n") if line.strip())
    return text

def get_dataset_text():
    all_text = []
    for filename in DATASOURCE:
        text = preprocess_gutenberg(f"{filename}.txt")
        all_text.append(text)
    return all_text

In [4]:
# Check what you will get
# text = get_dataset_text()

In [5]:
#text[2]

# Lesson 2: Train a Tokenizer for Your Language Model

In [6]:
# Tokenization with BPE (Byte-pair encoding): chooses the most frequent adjacent pair.
if os.path.exists("gutenberg_tokenizer.json"):
    tokenizer = tokenizers.Tokenizer.from_file("gutenberg_tokenizer.json")
else:
    tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE())
    # Configure pre-tokenizer add space at beginning of the sentence
    tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel(add_prefix_space=True)
    # Configure decoder so that the boundary symbols will be removed
    tokenizer.decoder = tokenizers.decoders.ByteLevel()
    # Train BPE
    VOCAB_SIZE = 10000
    trainer = tokenizers.trainers.BpeTrainer(
        vocab_size=VOCAB_SIZE,
        special_tokens=["[pad]", "[eos]"],
        show_progress=True
    )
    text = get_dataset_text()
    tokenizer.train_from_iterator(text, trainer=trainer)
    tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[pad]"), pad_token="[pad]")
    # Save the trained tokenizer
    tokenizer.save("gutenberg_tokenizer.json", pretty=True)

In [7]:
#tokenizer

## Why is a vocabulary size of 10,000 insufficient for a good language model? 

### Why Is a Vocabulary Size of 10,000 Often Insufficient?

A vocabulary size of 10,000 is often too small for a strong general-purpose language model because it may force the tokenizer to split many words into longer sequences of subword tokens.

A larger vocabulary can represent more common words, morphemes, and technical terms as individual tokens, improving tokenization efficiency.

With a smaller vocabulary, unrelated words may share subword fragments. Although subword sharing can help generalization, excessive fragmentation can increase sequence lengths.

#### 1. Embedding Matrix Size

A smaller vocabulary reduces the number of parameters in the token embedding matrix:

$$
P_{\text{embedding}} = V \times d
$$

where:

- $P_{\text{embedding}}$: Number of token embedding parameters
- $V$: Vocabulary size
- $d$: Embedding dimension

#### 2. Attention Computational Cost

Smaller vocabularies often produce longer token sequences, increasing the computational cost of standard full self-attention:

$$
O(n^2)
$$

where:

- $n$: Number of tokens in the input sequence

The quadratic term describes the scaling of attention interactions with sequence length.

#### 3. Vocabulary Sizes of Existing Models

Many successful language models use substantially larger vocabularies:

| Model | Approximate Vocabulary Size |
|---|---:|
| BERT Base | 30,522 |
| RoBERTa | 50,265 |
| GPT-2 | 50,257 |
| LLaMA 2 | 32,000 |
| Gemma (original) | 256,000 |

*Note: Vocabulary sizes depend on the specific model and tokenizer version.*

#### 4. Vocabulary Size Tradeoffs

Choosing an appropriate vocabulary size requires balancing:

1. **Embedding size:** Larger vocabularies require more embedding parameters.
2. **Sequence length:** Larger vocabularies can reduce token fragmentation and sequence length.
3. **Token efficiency:** Efficient tokenization represents text using fewer tokens.
4. **Language coverage:** Multilingual and specialized applications may benefit from larger vocabularies.

#### Conclusion

A vocabulary size of 10,000 can work for specialized or resource-constrained models, but it may limit tokenization efficiency and language coverage in general-purpose language models.

The optimal vocabulary size depends on the training corpus, tokenizer design, supported languages, and computational budget.


# Lesson 3: Position Encoding

Unlike recurrent neural networks, transformer models process entire sequences simultaneously. However, this parallel processing means they lack inherent understanding of token order. Since token position is crucial for understanding context, transformer models incorporate positional encodings into their input processing to capture this sequential information.

Rotary Positional Encoding (RoPE) has emerged as the most widely used approach. RoPE operates by applying rotational transformations to the embedded token vectors. Each token is represented as a vector, and the encoding process involves multiplying pairs of vector elements by a 2 ×2 rotation matrix:
$$ \hat{\mathbf{x}}_m = \begin{bmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{bmatrix} \mathbf{x}_m $$

In [8]:
def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(x, cos, sin):
    return (x * cos) + (rotate_half(x) * sin)

class RotaryPositionalEncoding(nn.Module):
    def __init__(self, dim, max_seq_len=1024):
        super().__init__()
        N = 10000
        inv_freq = 1. / (N ** (torch.arange(0, dim, 2).float() / dim))
        position = torch.arange(max_seq_len).float()
        inv_freq = torch.cat((inv_freq, inv_freq), dim=-1)
        sinusoid_inp = torch.outer(position, inv_freq)
        # Need register buffer, not just self.cos = x: Otherwise you can't let PyTorch move your tensor to GPU
        #self.cos = sinusoid_inp.cos()
        #self.sin = sinusoid_inp.sin()
        self.register_buffer("cos", sinusoid_inp.cos())
        self.register_buffer("sin", sinusoid_inp.sin())

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.size(-2)
        cos = self.cos[:seq_len].view(1, 1, seq_len, -1)
        sin = self.sin[:seq_len].view(1, 1, seq_len, -1)
        return apply_rotary_pos_emb(x, cos, sin)



# Rotary Positional Encoding (RoPE) and 4D Tensor Structure

## 1. Rotary Positional Encoding

The `RotaryPositionalEncoding` module implements rotary positional encoding (RoPE) for input sequences.

- **`__init__()`**: Precomputes sine and cosine values for supported positions and dimensions.
- **`forward()`**: Applies position-dependent rotations to the input tensor.

### Important Implementation Detail: `register_buffer`

The `register_buffer()` method stores precomputed sine and cosine tensors as **non-trainable buffers** rather than trainable model parameters.

This ensures that:

- The buffers move with the model between devices (e.g., CPU and GPU).
- They are included in model serialization through `state_dict()` by default.
- They are not updated by the optimizer.

---

## 2. Why Is the Input a 4D Tensor?

Consider the input tensor:

```python
x.shape = (1, 10, 4, 128)
```

In this implementation, the dimensions represent:

| Axis | Size | Meaning |
|---|---:|---|
| 0 | 1 | Batch size |
| 1 | 10 | Sequence length |
| 2 | 4 | Number of attention heads |
| 3 | 128 | Dimensions per attention head |

The general structure is:

$$
X \in \mathbb{R}^{B \times L \times H \times D_h}
$$

where:

- $B$: Batch size
- $L$: Sequence length (number of tokens)
- $H$: Number of attention heads
- $D_h$: Dimension of each attention head

For this example:

$$
X \in \mathbb{R}^{1 \times 10 \times 4 \times 128}
$$

### 2.1 Batch Size = 1

```python
x.shape[0] == 1
```

The batch contains one sequence:

```text
Batch 0: [Token 0, Token 1, ..., Token 9]
```

### 2.2 Sequence Length = 10

```python
x.shape[1] == 10
```

The sequence contains 10 token positions:

$$
p \in \{0,1,2,\ldots,9\}
$$

RoPE applies position-dependent rotation angles, so tokens at different positions receive different rotations.

### 2.3 Number of Attention Heads = 4

```python
x.shape[2] == 4
```

The representation contains four attention heads.

Multi-head attention processes information through separate head representations. RoPE applies the same positional rotation rule independently to each head.

For example, at token position 5:

```text
Token 5
├── Head 0 → 128-dimensional vector
├── Head 1 → 128-dimensional vector
├── Head 2 → 128-dimensional vector
└── Head 3 → 128-dimensional vector
```

All four vectors use rotation angles determined by token position 5, while their actual vector values may differ.

### 2.4 Head Dimension = 128

```python
x.shape[3] == 128
```

Each attention head has 128 feature dimensions.

In the adjacent-pair formulation, RoPE rotates pairs of dimensions:

```text
(x₀,   x₁)
(x₂,   x₃)
(x₄,   x₅)
...
(x₁₂₆, x₁₂₇)
```

This gives:

$$
\frac{128}{2} = 64
$$

rotation pairs per head.

For a pair of features, the rotation is:

$$
\begin{bmatrix}
x'_{2i} \\
x'_{2i+1}
\end{bmatrix}
=
\begin{bmatrix}
\cos(p\theta_i) & -\sin(p\theta_i) \\
\sin(p\theta_i) & \cos(p\theta_i)
\end{bmatrix}
\begin{bmatrix}
x_{2i} \\
x_{2i+1}
\end{bmatrix}
$$

where:

- $p$: Token position
- $i$: Feature-pair index
- $\theta_i$: Rotation frequency for pair $i$
- $x_{2i}, x_{2i+1}$: Original feature values
- $x'_{2i}, x'_{2i+1}$: Rotated feature values

**Note:** Some implementations pair dimensions differently, such as splitting the head dimension into two halves. The exact pairing must match the implementation.

---

## 3. Visualizing the 4D Tensor

**Tensor shape: `(1, 10, 4, 128)`**

```text
Batch 0
│
├── Token 0
│   ├── Head 0 : [128 values]
│   ├── Head 1 : [128 values]
│   ├── Head 2 : [128 values]
│   └── Head 3 : [128 values]
│
├── Token 1
│   ├── Head 0 : [128 values]
│   ├── Head 1 : [128 values]
│   ├── Head 2 : [128 values]
│   └── Head 3 : [128 values]
│
├── ...
│
└── Token 9
    ├── Head 0 : [128 values]
    ├── Head 1 : [128 values]
    ├── Head 2 : [128 values]
    └── Head 3 : [128 values]
```

Each token has four attention-head representations, and each head contains 128 values.

---

## 4. Why Is RoPE Applied to a 4D Tensor?

RoPE is typically applied to the **query (Q)** and **key (K)** tensors after projection into attention heads.

For this example:

$$
Q \in \mathbb{R}^{1 \times 10 \times 4 \times 128}
$$

$$
K \in \mathbb{R}^{1 \times 10 \times 4 \times 128}
$$

The positional rotation is applied:

1. At each of the 10 token positions.
2. Independently to each of the 4 attention heads.
3. Across pairs of features within each 128-dimensional head.

Thus, a common tensor layout for applying RoPE is:

$$
\boxed{
\text{Batch} \times
\text{Sequence} \times
\text{Heads} \times
\text{Head Dimension}
}
$$

For this example:

$$
\boxed{1 \times 10 \times 4 \times 128}
$$

**Important:** RoPE does not inherently require a 4D tensor. The shape depends on the implementation. Some transformer implementations use the alternative layout:

```python
(batch_size, num_heads, seq_len, head_dim)
```

Both layouts are valid when the positional dimensions and broadcasting are handled correctly.

---

## 5. Relationship to the Model Hidden Size

For standard multi-head attention, when the concatenated head dimension equals the model hidden dimension:

$$
d_{\text{model}} = H \times D_h
$$

where:

- $d_{\text{model}}$: Model hidden dimension
- $H$: Number of attention heads
- $D_h$: Dimension per head

In this example:

$$
d_{\text{model}} = 4 \times 128 = 512
$$

Therefore, the four 128-dimensional heads together form a 512-dimensional representation.

This calculation assumes a standard multi-head attention configuration. In some architectures, the number or dimensions of query and key/value heads differ.

---

## 6. Position of RoPE in the Attention Pipeline

A simplified transformer attention pipeline is:

```text
Input Hidden States
        │
        ▼
Linear Projections
   Q, K, V
        │
        ▼
Reshape into Attention Heads
        │
        ├── Q: (1, 10, 4, 128)
        ├── K: (1, 10, 4, 128)
        └── V: (1, 10, 4, 128)
        │
        ▼
Apply RoPE to Q and K
        │
        ▼
Compute Attention Scores
        │
        ▼
Softmax
        │
        ▼
Weighted Sum of V
        │
        ▼
Concatenate Head Outputs
        │
        ▼
Output Projection
```

The attention operation is:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{\widetilde{Q}\widetilde{K}^{T}}{\sqrt{D_h}}
\right)V
$$

where:

- $\widetilde{Q}$: Query tensor after RoPE
- $\widetilde{K}$: Key tensor after RoPE
- $V$: Value tensor (normally not rotated)
- $D_h$: Attention head dimension
- $T$: Matrix transpose over the token and feature dimensions within each head

The matrix multiplication and softmax are computed independently for each attention head.

---

## Key Takeaways

- The tensor `(1, 10, 4, 128)` represents **batch size, sequence length, attention heads, and dimensions per head**.
- RoPE introduces positional information by rotating feature pairs in query and key vectors.
- RoPE uses token positions to determine rotation angles, with separate frequencies for different feature pairs.
- For standard multi-head attention, 4 heads of dimension 128 correspond to a concatenated dimension of 512.
- RoPE is applied **before calculating attention scores**, enabling attention scores to encode relative positional information.


# Lesson 4: Grouped Query Attention

The signature component of a transformer model is its attention mechanism. When processing a sequence of tokens, the attention mechanism builds connections between tokens to understand their context.

The attention mechanism predates transformer models, and several variants have evolved over time. In this lesson, you will learn to implement Grouped Query Attention (GQA).

A transformer model begins with a sequence of embedded tokens, which are essentially vectors. The modern attention mechanism computes an output sequence based on three input sequences: query, key, and value. These three sequences are derived from the input sequence through different projections:
        
The projection is performed by a fully-connected neural network layer that operates on the input tensor’s last dimension. As shown above, the projection’s output is reshaped using view() and then transposed. The input tensor x is 3D, and the view() function transforms it into a 4D tensor by splitting the last dimension into two: the attention heads and the head dimension. The transpose() function then swaps the sequence length dimension with the attention head dimension.

The resulting 4D tensor has attention operations that only involve the last two dimensions. The actual attention computation is performed using PyTorch’s built-in scaled_dot_product_attention() function. The result is then reshaped back into a 3D tensor and projected to the original dimension

This architecture is called grouped query attention because it uses different numbers of heads for queries versus keys and values. Typically, the number of query heads is a multiple of the number of key-value heads.

Since we will use such attention mechanism a lot, let’s create a class for it:

In [9]:
'''
Multi-head attention: Each query head attends to its corresponding key and value head:

headi=Attention(Qi,Ki,Vi)\text{head}_i = \text{Attention}(Q_i, K_i, V_i)headi​=Attention(Qi​,Ki​,Vi​)

Grouped Query Attention: Multiple query heads share the same key/value head:
Q0, Q1, Q2, Q3 --> KV0
Q4, Q5, Q6, Q7 --> KV1
This reduces memory usage because fewer KKK and VVV tensors must be stored during inference.

Convert GQA into MHA: \n
1. remove: self.num_kv_heads = num_kv_heads \n
           self.num_groups = num_heads // num_kv_heads  \n
2. edit to: def __init__(self, hidden_dim, num_heads, dropout=0.1):
3. edit to: self.k_proj = nn.Linear(
    hidden_dim,
    self.num_heads * self.head_dim
)

4. self.v_proj = nn.Linear(
    hidden_dim,
    self.num_heads * self.head_dim
)

5. attention call: output = F.scaled_dot_product_attention(
    q,
    k,
    v,
    attn_mask=mask,
    dropout_p=self.dropout
)

'''

class GQA(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_kv_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads ## 
        self.head_dim = hidden_dim // num_heads
        self.num_groups = num_heads // num_kv_heads
        self.dropout = dropout
        self.q_proj = nn.Linear(hidden_dim, self.num_heads * self.head_dim)
        self.k_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(hidden_dim, self.num_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(self.num_heads * self.head_dim, hidden_dim)

    def forward(self, q, k, v, mask=None, rope=None):
        q_batch_size, q_seq_len, hidden_dim = q.shape
        k_batch_size, k_seq_len, hidden_dim = k.shape
        v_batch_size, v_seq_len, hidden_dim = v.shape

        # projection
        q = self.q_proj(q).view(q_batch_size, q_seq_len, -1, self.head_dim).transpose(1, 2)
        k = self.k_proj(k).view(k_batch_size, k_seq_len, -1, self.head_dim).transpose(1, 2)
        v = self.v_proj(v).view(v_batch_size, v_seq_len, -1, self.head_dim).transpose(1, 2)

        # apply rotary positional encoding
        if rope:
            q = rope(q)
            k = rope(k)

        # compute grouped query attention
        q = q.contiguous()
        k = k.contiguous()
        v = v.contiguous()
        output = F.scaled_dot_product_attention(q, k, v,
                                                attn_mask=mask,
                                                dropout_p=self.dropout if self.training else 0.0,
                                                enable_gqa=True)
        output = output.transpose(1, 2).reshape(q_batch_size, q_seq_len, hidden_dim).contiguous()
        output = self.out_proj(output)
        return output

In [10]:
# class MultiHeadAttention(nn.Module):
#     def __init__(self, hidden_dim, num_heads, dropout=0.1):
#         super().__init__()

#         self.num_heads = num_heads
#         self.head_dim = hidden_dim // num_heads
#         self.dropout = dropout

#         self.q_proj = nn.Linear(
#             hidden_dim,
#             num_heads * self.head_dim
#         )

#         self.k_proj = nn.Linear(
#             hidden_dim,
#             num_heads * self.head_dim
#         )

#         self.v_proj = nn.Linear(
#             hidden_dim,
#             num_heads * self.head_dim
#         )

#         self.out_proj = nn.Linear(
#             num_heads * self.head_dim,
#             hidden_dim
#         )

#     def forward(self, q, k, v, mask=None, rope=None):

#         batch_size, q_seq_len, hidden_dim = q.shape
#         _, k_seq_len, _ = k.shape
#         _, v_seq_len, _ = v.shape

#         # projections
#         q = self.q_proj(q).view(
#             batch_size,
#             q_seq_len,
#             self.num_heads,
#             self.head_dim
#         ).transpose(1, 2)

#         k = self.k_proj(k).view(
#             batch_size,
#             k_seq_len,
#             self.num_heads,
#             self.head_dim
#         ).transpose(1, 2)

#         v = self.v_proj(v).view(
#             batch_size,
#             v_seq_len,
#             self.num_heads,
#             self.head_dim
#         ).transpose(1, 2)

#         # RoPE applied per head
#         if rope is not None:
#             q = rope(q)
#             k = rope(k)

#         output = F.scaled_dot_product_attention(
#             q,
#             k,
#             v,
#             attn_mask=mask,
#             dropout_p=self.dropout,
#             is_causal=False
#         )

#         output = (
#             output.transpose(1, 2)
#             .reshape(batch_size, q_seq_len, hidden_dim)
#         )

#         output = self.out_proj(output)

#         return output

The forward function includes two optional arguments: mask and rope. The rope argument expects a module that applies rotary positional encoding, which was covered in the previous lesson. The mask argument will be explained in the next lesson.



# Lesson 5: Causal Masks


# Causal Masks in Decoder-Only Transformers

A key characteristic of **decoder-only transformer models** is the use of **causal masks** in their attention layers.

A causal mask is a matrix applied during attention score calculation to prevent the model from attending to future tokens.

Specifically, a query token $i$ can only attend to key tokens $j$ where

$$
\boxed{j \leq i}
$$

This ensures that each token can only access itself and preceding tokens.

## 1. Causal Mask Structure

With query and key sequences of length $N$, the causal mask is a square matrix of shape

$$
(N, N)
$$

The element $(i,j)$ indicates whether query token $i$ can attend to key token $j$.

### Boolean Causal Mask

In a Boolean mask where `True` means attention is allowed:

$$
M_{ij} =
\begin{cases}
\text{True}, & j \leq i \\
\text{False}, & j > i
\end{cases}
$$

For a sequence of length $N=4$:

```text
          Key Position
          0  1  2  3
Query  0  T  F  F  F
       1  T  T  F  F
       2  T  T  T  F
       3  T  T  T  T
```

Here, `T` means attention is allowed, and `F` means attention is blocked.

### Additive Floating-Point Causal Mask

We can alternatively use a floating-point mask and add it to the attention scores before applying softmax.

$$
M_{ij} =
\begin{cases}
0, & j \leq i \\
-\infty, & j > i
\end{cases}
$$

For $N=4$:

$$
M =
\begin{bmatrix}
0 & -\infty & -\infty & -\infty \\
0 & 0 & -\infty & -\infty \\
0 & 0 & 0 & -\infty \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

The value $-\infty$ ensures that masked positions receive zero attention weight after softmax.

---

## 2. Creating a Causal Mask in PyTorch

Creating an additive causal mask is straightforward:

```python
import torch

N = 4

mask = torch.triu(
    torch.full((N, N), float("-inf")),
    diagonal=1
)

print(mask)
```

**Explanation:**

1. `torch.full((N, N), float("-inf"))` creates an $N \times N$ matrix filled with $-\infty$.
2. `torch.triu(..., diagonal=1)` retains elements strictly above the main diagonal.
3. All elements on and below the diagonal become zero.

The resulting matrix is:

```text
tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])
```

---

## 3. Applying the Causal Mask to Attention

The scaled dot-product attention equation with an additive causal mask is:

$$
\boxed{
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}} + M
\right)V
}
$$

where:

- $Q$: Query matrix
- $K$: Key matrix
- $V$: Value matrix
- $d_k$: Dimension of the key vectors
- $M$: Additive causal mask containing $0$ and $-\infty$

### PyTorch Implementation

```python
import torch.nn.functional as F

output = F.scaled_dot_product_attention(
    q,
    k,
    v,
    attn_mask=mask
)
```

The mask is added to the attention scores before softmax, preventing attention to future positions.

**Alternative:** When only causal masking is needed, PyTorch provides a built-in option:

```python
output = F.scaled_dot_product_attention(
    q,
    k,
    v,
    is_causal=True
)
```

This avoids explicitly constructing the causal mask.

---

## 4. Combining Causal and Padding Masks

Sometimes we must also mask additional positions, such as padding tokens.

A padding mask prevents the model from attending to tokens introduced only to make sequences in a batch have the same length.

For additive masks, disallowed key positions receive $-\infty$.

### Mask Dimensions

For multi-head attention, query and key tensors commonly have the shapes:

$$
Q \in \mathbb{R}^{B \times H \times L \times D_k}
$$

$$
K \in \mathbb{R}^{B \times H \times S \times D_k}
$$

where:

- $B$: Batch size
- $H$: Number of attention heads
- $L$: Query sequence length
- $S$: Key sequence length
- $D_k$: Dimension per head

The attention score tensor has shape:

$$
(B,H,L,S)
$$

**Causal mask shared across the batch:**

$$
M_{\text{causal}} \in \mathbb{R}^{L \times S}
$$

**Combined causal and padding mask (per batch element):**

$$
M_{\text{combined}} \in \mathbb{R}^{B \times 1 \times L \times S}
$$

The singleton head dimension allows the mask to broadcast across all attention heads.

A mask of shape $(B,H,L,S)$ can also be used when each attention head requires its own mask.

**Note:** Although a batch-specific mask can conceptually be represented as $(B,L,S)$, this shape does not generally broadcast correctly against PyTorch's standard attention-score layout $(B,H,L,S)$ when multiple heads are present.

---

## 5. Key Takeaways

1. **Causal masking** prevents tokens from attending to future positions.
2. Query token $i$ may attend only to key tokens satisfying $j \leq i$.
3. A causal mask for a sequence of length $N$ has shape $(N,N)$.
4. In an additive mask, allowed positions contain $0$, while blocked positions contain $-\infty$.
5. `torch.triu(..., diagonal=1)` can construct an additive causal mask.
6. `F.scaled_dot_product_attention()` supports explicit masks through `attn_mask` and built-in causal masking through `is_causal=True`.
7. When combining causal and padding masks, the mask must be broadcastable to the attention-score tensor.

**Core principle:** Causal masking preserves the autoregressive property of decoder-only transformers by ensuring each prediction depends only on permitted preceding context.


In [11]:
def create_causal_mask(seq_len, device, dtype=torch.float32):
    """Create a causal mask for autoregressive attention."""
    mask = torch.triu(torch.full((seq_len, seq_len), float('-inf'), device=device, dtype=dtype), diagonal=1)
    return mask


## Question: Combining Causal and Padding Masks

Given the `scaled_dot_product_attention()` call and a query tensor `q` of shape $(B,H,N,D)$ containing some padding tokens, how would you create a mask tensor of shape $(B,N,N)$ that combines both causal and padding masks to:

1. **Prevent attention to future tokens.**
2. **Mask all attention operations involving padding tokens.**

## Answer

We can combine a **causal mask** and a **padding mask** using a logical AND operation.

### 1. Masking Rules

For batch element $b$, query position $i$, and key position $j$, attention is allowed only when:

$$
\boxed{
M_{b,i,j} = (j \leq i) \land V_{b,i} \land V_{b,j}
}
$$

where:

- $M_{b,i,j}$: Boolean attention mask.
- $j \leq i$: Prevents attending to future tokens.
- $V_{b,i}$: `True` if the query token is not padding.
- $V_{b,j}$: `True` if the key token is not padding.
- $\land$: Logical AND.

### 2. PyTorch Implementation

Assume `q` has shape `(B, H, N, D)` and `valid_tokens` is a Boolean tensor of shape `(B, N)`, where `True` indicates a non-padding token.

The validity mask should normally come from the original token IDs or attention mask, rather than being inferred from `q`.

```python id="qe9az2"
import torch

B, H, N, D = q.shape

# valid_tokens: (B, N), True for non-padding tokens
# Example:
# valid_tokens = input_ids.ne(pad_token_id)

# 1. Causal mask: (N, N)
causal_mask = torch.tril(
    torch.ones(N, N, dtype=torch.bool, device=q.device)
)

# 2. Padding masks
query_mask = valid_tokens[:, :, None]  # (B, N, 1)
key_mask = valid_tokens[:, None, :]    # (B, 1, N)

# 3. Combine causal and padding masks
mask = causal_mask[None, :, :] & query_mask & key_mask

assert mask.shape == (B, N, N)
```

**Result:**

$$
\boxed{\operatorname{shape}(M) = (B,N,N)}
$$

### 3. Example

Suppose a sequence contains five positions:

```text id="wr0h1m"
Tokens:       A   B   C   PAD PAD
Valid tokens: T   T   T    F   F
```

The combined mask is:

```text id="l8sdeu"
              Key positions
              0  1  2  3  4
Query  0      T  F  F  F  F
       1      T  T  F  F  F
       2      T  T  T  F  F
       3      F  F  F  F  F
       4      F  F  F  F  F
```

- **Causal masking:** Blocks positions above the diagonal.
- **Padding-key masking:** Blocks columns corresponding to padding tokens.
- **Padding-query masking:** Blocks rows corresponding to padding tokens.

### 4. Applying the Mask to Attention

Although the requested mask has shape `(B,N,N)`, PyTorch's scaled dot-product attention typically expects a mask broadcastable to the attention scores of shape `(B,H,N,N)`.

Add a singleton head dimension:

```python id="vnclnm"
# (B, N, N) -> (B, 1, N, N)
attention_mask = mask.unsqueeze(1)
```

This allows broadcasting across all attention heads.

**Important:** Fully masked padding-query rows can cause numerical issues in some attention implementations. A robust approach is to mask causal and padding-key positions during attention, then zero the output at padding-query positions.

```python id="d1hozu"
import torch.nn.functional as F

# (B, 1, N, N): blocks future and padding keys
safe_mask = (
    causal_mask[None, None, :, :]
    & valid_tokens[:, None, None, :]
)

output = F.scaled_dot_product_attention(
    q, k, v,
    attn_mask=safe_mask,
    dropout_p=0.0
)

# Exclude padded query positions from the output
output = output.masked_fill(
    ~valid_tokens[:, None, :, None],
    0.0
)
```

This approach assumes every query position has at least one permitted non-padding key, as in ordinary right-padded causal sequences with at least one valid token.

### Key Takeaways

- The **causal mask** prevents attention to future tokens.
- The **padding mask** excludes padding tokens as queries and keys.
- Logical AND combines both conditions into a Boolean mask of shape $(B,N,N)$.
- Adding a head dimension gives $(B,1,N,N)$ for broadcasting in multi-head attention.
- For numerical stability, padded query outputs can be zeroed after attention instead of using fully masked attention rows.


# Lesson 6: Mixture of Expert Models

Transformer models consist of stacked transformer blocks, where each block contains an attention sublayer and an MLP sublayer. The attention sublayer implements a multi-head attention mechanism, while the MLP sublayer is a feed-forward network.

The MLP sublayer introduces non-linearity to the model and is where much of the model’s “intelligence” resides. To enhance the model’s capabilities, you can either increase the size of the feed-forward network or employ a more sophisticated architecture like Mixture of Experts (MoE).

MoE is a recent innovation in transformer models. It consists of multiple parallel MLP sublayers with a router that selects a subset of them to process the input. The final output is a weighted sum of the outputs from the selected MLP sublayers. Many modern large language models use SwiGLU as their MLP sublayer, which combines three linear transformations with a SiLU activation function. Here’s how to implement it:

For example, in a system with 8 MLP sublayers, the router processes each input token using a linear layer to produce 8 scores. The top 2 scoring sublayers are selected to process the input, and their outputs are combined using weighted summation.

In [12]:
class SwiGLU(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim):
        super().__init__()
        self.gate = nn.Linear(hidden_dim, intermediate_dim)
        self.up = nn.Linear(hidden_dim, intermediate_dim)
        self.down = nn.Linear(intermediate_dim, hidden_dim)
        self.act = nn.SiLU()

    def forward(self, x):
        x = self.act(self.gate(x)) * self.up(x)
        x = self.down(x)
        return x

class MoELayer(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim, num_experts, top_k=2):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        # Create expert networks
        self.experts = nn.ModuleList([
            SwiGLU(hidden_dim, intermediate_dim) for _ in range(num_experts)
        ])
        self.router = nn.Linear(hidden_dim, num_experts)

    def forward(self, hidden_states):
        batch_size, seq_len, hidden_dim = hidden_states.shape

        # Reshape for expert processing, then compute routing probabilities
        hidden_states_reshaped = hidden_states.view(-1, hidden_dim)
        # shape of router_logits: (batch_size * seq_len, num_experts)
        router_logits = self.router(hidden_states_reshaped)

        # Select top-k experts, then softmax output probabilities will sum to 1
        # output shape: (batch_size * seq_len, k)
        top_k_logits, top_k_indices = torch.topk(router_logits, self.top_k, dim=-1)
        top_k_probs = F.softmax(top_k_logits, dim=-1)

        # Allocate output tensor
        output = torch.zeros(batch_size * seq_len, hidden_dim,
                             device=hidden_states.device,
                             dtype=hidden_states.dtype)

        # Process through selected experts
        unique_experts = torch.unique(top_k_indices)
        for i in unique_experts:
            expert_id = int(i)
            # token_mask (boolean tensor) = which token of the input should use this expert
            # token_mask shape: (batch_size * seq_len,)
            mask = (top_k_indices == expert_id)
            token_mask = mask.any(dim=1)
            assert token_mask.any(), f"Expecting some tokens using expert {expert_id}"

            # select tokens, apply the expert, then add to the output
            expert_input = hidden_states_reshaped[token_mask]
            expert_weight = top_k_probs[mask].unsqueeze(-1)       # shape: (N, 1)
            expert_output = self.experts[expert_id](expert_input) # shape: (N, hidden_dim)
            output[token_mask] += expert_output * expert_weight

        # Reshape back to original shape
        output = output.view(batch_size, seq_len, hidden_dim)
        return output


## Mixture of Experts (MoE): Forward Pass

The `forward()` method implements the **Mixture of Experts (MoE)** routing and computation process.

### 1. Expert Selection

The router first generates two outputs:

- **`top_k_indices`**: Indices of the top-$k$ experts selected for each token.
- **`top_k_probs`**: Routing weights assigned to the selected experts.

These determine which experts process each token and how much each expert contributes to the final output.

### 2. Expert Computation

Based on `top_k_indices`, the method selects and applies the corresponding experts to the input tokens.

The input is a 3D tensor:

$$
X \in \mathbb{R}^{B \times L \times D}
$$

where:

- $B$: Batch size (`batch_size`)
- $L$: Sequence length (`seq_len`)
- $D$: Hidden dimension (`hidden_dim`)

Different tokens within the same sequence may be processed by different experts.

### 3. Weighted Combination of Expert Outputs

The selected expert outputs are combined using their routing weights (`top_k_probs`):

$$
\boxed{
y_t = \sum_{i=1}^{k} p_{t,i} E_{e_{t,i}}(x_t)
}
$$

where:

- $x_t$: Input representation of token $t$
- $y_t$: Final MoE output for token $t$
- $k$: Number of selected experts per token
- $e_{t,i}$: Index of the $i$-th selected expert for token $t$
- $E_{e_{t,i}}(x_t)$: Output produced by the selected expert
- $p_{t,i}$: Routing weight for that expert

The equation assumes routing weights are used directly in a weighted sum.

### 4. Masking During Expert Processing

Because different tokens may be assigned to different experts, the method uses **masking** to identify which tokens each expert should process.

The general process is:

1. Identify tokens assigned to each expert using `top_k_indices`.
2. Apply the corresponding expert only to its selected tokens.
3. Multiply each expert output by its routing weight from `top_k_probs`.
4. Accumulate the weighted outputs at the original token positions.

### 5. Example

Suppose a token is routed to two experts:

| Selected Expert | Routing Weight |
|---|---:|
| Expert 1 | 0.7 |
| Expert 3 | 0.3 |

The combined output is:

$$
y_t = 0.7 E_1(x_t) + 0.3 E_3(x_t)
$$

This allows the model to combine specialized computations from multiple experts without executing every expert for every token.

### Key Takeaway

The MoE `forward()` method performs three main operations:

$$
\boxed{
\text{Router Selection}
\rightarrow
\text{Expert Computation}
\rightarrow
\text{Weighted Aggregation}
}
$$

**Masking ensures that each expert processes its assigned tokens and that the weighted results are accumulated at the correct positions in the output tensor.**


Q:Models like DeepSeek V2 incorporate a shared expert in their MoE architecture. It is an expert that processes every input regardless of routing. Can you modify the code above to include a shared expert?  

# Mixture of Experts (MoE): Adding a Shared Expert

## Question

How can we modify an MoE layer so that **one shared expert processes every input token**, while a router selects the top-$k$ experts from the remaining routed experts?

## 1. Main idea

The shared expert runs on **all tokens**, regardless of router decisions. The top-$k$ routed experts process only their assigned tokens. The two contributions are added together:

$$
\boxed{y_t = E_{\mathrm{shared}}(x_t) + \sum_{r=1}^{k} p_{t,r} E_{e_{t,r}}(x_t)}
$$

where:

- $x_t \in \mathbb{R}^{D}$: Input representation of token $t$.
- $E_{\mathrm{shared}}$: Shared feed-forward expert applied to every token.
- $e_{t,r}$: Index of the $r$-th selected routed expert for token $t$.
- $E_{e_{t,r}}$: Selected routed expert.
- $p_{t,r}$: Softmax-normalized routing weight for that expert.
- $k$: Number of routed experts selected per token.
- $y_t \in \mathbb{R}^{D}$: Output token representation.

In this illustrative implementation, $\sum_{r=1}^{k} p_{t,r}=1$ **among routed experts only**. The shared expert is not included in the routing softmax.

## 2. Routing diagram

![Shared expert and top-k routed experts](attachment:shared_expert_moe_diagram.png)

For example, if a token is routed to Expert 1 with weight $0.7$ and Expert 3 with weight $0.3$:

$$
y_t = E_{\mathrm{shared}}(x_t) + 0.7E_1(x_t) + 0.3E_3(x_t)
$$

## 3. Tensor shapes and steps

Given input $X \in \mathbb{R}^{B\times L\times D}$, flatten its first two dimensions so $T=B\,L$:

| Quantity | Shape | Purpose |
|---|---|---|
| Input `hidden_states` | `(B, L, D)` | Batch of token representations |
| Flattened input `x` | `(T, D)` | Tokens processed independently |
| Router logits | `(T, num_experts)` | Score each routed expert |
| `top_k_indices` | `(T, k)` | Selected routed experts per token |
| `top_k_probs` | `(T, k)` | Normalized weights of selected experts |
| Shared output | `(T, D)` | Contribution for **every** token |
| Routed output | `(T, D)` | Weighted sum over selected experts |
| Final output | `(B, L, D)` | Original tensor shape restored |

The complete example below implements the `SwiGLU` feed-forward expert, `MoELayer` with one shared expert, and forward/backward-pass checks **in a single Python cell**.

## 4. Implementation notes

- `self.shared_expert` is separate from `self.experts` and does **not** participate in `topk` selection.
- `torch.where` identifies the token assignments for each routed expert; `index_add_` accumulates the weighted outputs at the original token indices.
- This is an **educational shared-expert MoE**, not a full reimplementation of DeepSeek-V2; production variants can include additional routing, capacity, and load-balancing mechanisms.
- The sum of routed probabilities is 1, but adding the shared output means the **combined output is not a normalized average**.


In [13]:
# ## shared expert

# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# # ------------------------------------------------------------
# # 1. SwiGLU feed-forward expert
# # ------------------------------------------------------------
# class SwiGLU(nn.Module):
#     def __init__(self, hidden_dim: int, intermediate_dim: int):
#         super().__init__()
#         self.gate_proj = nn.Linear(hidden_dim, intermediate_dim)
#         self.up_proj = nn.Linear(hidden_dim, intermediate_dim)
#         self.down_proj = nn.Linear(intermediate_dim, hidden_dim)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


# # ------------------------------------------------------------
# # 2. MoE with one always-on shared expert
# # ------------------------------------------------------------
# class MoELayer(nn.Module):
#     def __init__(
#         self,
#         hidden_dim: int,
#         intermediate_dim: int,
#         num_experts: int,
#         top_k: int = 2,
#     ):
#         super().__init__()
#         if not (1 <= top_k <= num_experts):
#             raise ValueError('Expected 1 <= top_k <= num_experts')

#         self.num_experts = num_experts
#         self.top_k = top_k

#         # Routed experts: only selected tokens are processed.
#         self.experts = nn.ModuleList([
#             SwiGLU(hidden_dim, intermediate_dim)
#             for _ in range(num_experts)
#         ])

#         # Shared expert: processes every token (not part of router).
#         self.shared_expert = SwiGLU(hidden_dim, intermediate_dim)
#         self.router = nn.Linear(hidden_dim, num_experts)

#     def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
#         # hidden_states: (B, L, D)
#         batch_size, seq_len, hidden_dim = hidden_states.shape
#         x = hidden_states.reshape(-1, hidden_dim)  # (T, D), T = B * L

#         # Always process all tokens through the shared expert.
#         shared_output = self.shared_expert(x)  # (T, D)

#         # Select the top-k routed experts for each token.
#         router_logits = self.router(x)  # (T, num_experts)
#         top_k_logits, top_k_indices = torch.topk(
#             router_logits, k=self.top_k, dim=-1
#         )  # (T, k), (T, k)
#         top_k_probs = F.softmax(top_k_logits, dim=-1)  # (T, k)

#         # Accumulate the weighted routed-expert contributions.
#         routed_output = torch.zeros_like(x)  # (T, D)
#         for expert_id, expert in enumerate(self.experts):
#             token_indices, slot_indices = torch.where(top_k_indices == expert_id)
#             if token_indices.numel() == 0:
#                 continue

#             expert_input = x[token_indices]                   # (selected, D)
#             expert_weight = top_k_probs[
#                 token_indices, slot_indices
#             ].unsqueeze(-1)                                # (selected, 1)
#             expert_output = expert(expert_input)             # (selected, D)
#             routed_output.index_add_(
#                 0, token_indices, expert_output * expert_weight
#             )

#         # Add shared contribution and restore original (B, L, D) shape.
#         output = (shared_output + routed_output).reshape(
#             batch_size, seq_len, hidden_dim
#         )
#         return output


# # ------------------------------------------------------------
# # 3. Fully runnable usage and validation
# # ------------------------------------------------------------
# torch.manual_seed(42)
# model = MoELayer(
#     hidden_dim=128,
#     intermediate_dim=256,
#     num_experts=4,
#     top_k=2,
# )

# # 2 sequences, 10 tokens each, 128 hidden features per token.
# x = torch.randn(2, 10, 128, requires_grad=True)
# y = model(x)

# print('Input shape: ', tuple(x.shape))
# print('Output shape:', tuple(y.shape))
# assert y.shape == x.shape

# # Backpropagate and verify that the shared expert and router train.
# loss = y.square().mean()
# loss.backward()
# assert x.grad is not None
# assert any(p.grad is not None for p in model.shared_expert.parameters())
# assert any(p.grad is not None for p in model.router.parameters())
# assert any(
#     p.grad is not None
#     for expert in model.experts
#     for p in expert.parameters()
# )
# print('Forward and backward checks passed.')

# # Inspect sample routing decisions and weights, without changing the model.
# with torch.no_grad():
#     logits = model.router(x.detach().reshape(-1, x.shape[-1]))
#     selected_logits, selected_ids = torch.topk(logits, k=model.top_k, dim=-1)
#     selected_weights = F.softmax(selected_logits, dim=-1)

# print('First token routed expert IDs:', selected_ids[0].tolist())
# print('First token routing weights:', selected_weights[0].tolist())
# print('Routing weight sum:', selected_weights[0].sum().item())


# Lesson 7: RMS Norm and Skip Connections

A Transformer is a typical deep learning model that can easily stack hundreds of transformer blocks, with each block containing multiple operations.  
Such deep models are sensitive to the vanishing gradient problem. Normalization layers are added to mitigate this issue and stabilize the training.

The two most common normalization layers in transformer models are Layer Norm and RMS Norm. We will use RMS Norm because it has fewer parameters. Using the built-in RMS Norm layer in PyTorch is straightforward:  
rms_norm = nn.RMSNorm(hidden_dim)  
output_rms = rms_norm(x)  

There are two ways to use RMS Norm in a transformer model: pre-norm and post-norm. In pre-norm, you apply RMS Norm before the attention and feed-forward sublayers, while in post-norm, you apply it after. This difference becomes clear when considering the skip connections. Here’s an example of a decoder-only transformer block with pre-norm:

In [14]:
class DecoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_heads, num_kv_heads, moe_experts, moe_topk, dropout=0.1):
        super().__init__()
        self.self_attn = GQA(hidden_dim, num_heads, num_kv_heads, dropout)
        self.mlp = MoELayer(hidden_dim, 4 * hidden_dim, moe_experts, moe_topk)
        self.norm1 = nn.RMSNorm(hidden_dim)
        self.norm2 = nn.RMSNorm(hidden_dim)

    def forward(self, x, mask=None, rope=None):
        # self-attention sublayer
        out = self.norm1(x)
        out = self.self_attn(out, out, out, mask, rope)
        x = out + x
        # MLP sublayer
        out = self.norm2(x)
        out = self.mlp(out)
        return out + x

Each transformer block contains an attention sublayer (implemented using the GQA class from lesson 4) and a feed-forward sublayer (implemented using the MoE class from lesson 6), along with two RMS Norm layers.

In the forward() method, we first normalize the input before applying the attention sublayer. Then, for the skip connection, we add the original unnormalized input to the attention sublayer’s output. In a post-norm approach, we would instead apply attention to the unnormalized input and then normalize the tensor after the skip connection. Research has shown that the pre-norm approach provides more stable training.

## From the description above, how would you modify the code to make it a post-norm transformer block?  


# Post-Norm Transformer Block

## 1. Pre-Norm vs. Post-Norm

Transformers commonly use two normalization strategies:

- **Pre-Norm:** Apply normalization before the attention or feed-forward sublayer.
- **Post-Norm:** Apply normalization after the sublayer output is added to the residual connection.

### Pre-Norm

$$
h = x + \operatorname{Attention}(\operatorname{RMSNorm}_1(x))
$$

$$
y = h + \operatorname{MoE}(\operatorname{RMSNorm}_2(h))
$$

### Post-Norm

$$
\boxed{
h = \operatorname{RMSNorm}_1(x + \operatorname{Attention}(x))
}
$$

$$
\boxed{
y = \operatorname{RMSNorm}_2(h + \operatorname{MoE}(h))
}
$$

where:

- $x$: Input hidden states
- $h$: Output of the attention sublayer and its residual connection after normalization
- $y$: Final Transformer block output
- $\operatorname{Attention}$: Grouped-Query Attention (GQA) sublayer
- $\operatorname{MoE}$: Mixture of Experts feed-forward sublayer
- $\operatorname{RMSNorm}$: Root Mean Square Normalization

## 2. Architecture Comparison

### Pre-Norm

```text
Input x
   |
   v
RMSNorm
   |
   v
Attention
   |
   v
Add Residual (x)
   |
   v
RMSNorm
   |
   v
MoE
   |
   v
Add Residual
   |
   v
Output
```

### Post-Norm

```text
Input x
   |
   v
Attention
   |
   v
Add Residual (x)
   |
   v
RMSNorm
   |
   v
MoE
   |
   v
Add Residual
   |
   v
RMSNorm
   |
   v
Output
```

## 3. Main Code Changes

**Original Pre-Norm:**

```python
out = self.norm1(x)
out = self.self_attn(out, out, out, mask, rope)
x = out + x

out = self.norm2(x)
out = self.mlp(out)
return out + x
```

**Modified Post-Norm:**

```python
out = self.self_attn(x, x, x, mask, rope)
x = self.norm1(x + out)

out = self.mlp(x)
x = self.norm2(x + out)
return x
```

The key change is the order of operations:

$$
\text{Pre-Norm: Normalization} \rightarrow
\text{Sublayer} \rightarrow \text{Residual Addition}
$$

$$
\text{Post-Norm: Sublayer} \rightarrow
\text{Residual Addition} \rightarrow \text{Normalization}
$$

## 4. Training Considerations

| Property | Pre-Norm | Post-Norm |
|---|---|---|
| Normalization location | Before sublayer | After residual addition |
| Residual path | Direct identity path | Passes through normalization |
| Gradient propagation | Generally easier in deep networks | Can be more challenging |
| Training stability | Often more stable at large depth | May require careful optimization |
| Output shape | Unchanged | Unchanged |

**Key Takeaway:** Post-norm changes the placement of normalization, not the attention mechanism, MoE routing, or tensor dimensions. Pre-norm is commonly favored in very deep Transformers because its residual structure generally improves gradient propagation and training stability.


In [15]:

# import torch
# import torch.nn as nn

# class DecoderLayer(nn.Module):
#     def __init__(
#         self,
#         hidden_dim,
#         num_heads,
#         num_kv_heads,
#         moe_experts,
#         moe_topk,
#         dropout=0.1
#     ):
#         super().__init__()

#         # Grouped-Query Attention
#         self.self_attn = GQA(
#             hidden_dim, num_heads, num_kv_heads, dropout
#         )

#         # Mixture of Experts feed-forward sublayer
#         self.mlp = MoELayer(
#             hidden_dim, 4 * hidden_dim, moe_experts, moe_topk
#         )

#         # RMS normalization layers
#         self.norm1 = nn.RMSNorm(hidden_dim)
#         self.norm2 = nn.RMSNorm(hidden_dim)

#     def forward(self, x, mask=None, rope=None):

#         # 1. Attention sublayer (Post-Norm)
#         # Attention operates on unnormalized input
#         out = self.self_attn(x, x, x, mask, rope)

#         # Residual addition followed by RMSNorm
#         x = self.norm1(x + out)

#         # 2. MoE feed-forward sublayer (Post-Norm)
#         out = self.mlp(x)

#         # Residual addition followed by RMSNorm
#         x = self.norm2(x + out)

#         return x


# Lesson 8: The Complete Transformer Model

So far, we have created all the building blocks of the transformer model. We can build a complete transformer model by stacking these blocks together. Before doing that, let’s list out the design parameters by creating a dictionary for the model configuration:

In [16]:
class TextGenerationModel(nn.Module):
    def __init__(self, num_layers, num_heads, num_kv_heads, hidden_dim,
                 moe_experts, moe_topk, max_seq_len, vocab_size, dropout=0.1):
        super().__init__()
        self.rope = RotaryPositionalEncoding(hidden_dim // num_heads, max_seq_len)
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.decoders = nn.ModuleList([
            DecoderLayer(hidden_dim, num_heads, num_kv_heads, moe_experts, moe_topk, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.RMSNorm(hidden_dim)
        self.out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, ids, mask=None):
        x = self.embedding(ids)
        for decoder in self.decoders:
            x = decoder(x, mask, self.rope)
        x = self.norm(x)
        return self.out(x)

In [17]:
# Training configuration
model_config = {
    "num_layers": 8,
    "num_heads": 8,
    "num_kv_heads": 4,
    "hidden_dim": 768,
    "moe_experts": 8,
    "moe_topk": 3,
    "max_seq_len": 512,
    "vocab_size": len(tokenizer.get_vocab()),
    "dropout": 0.1,
}

# Initialize model, optimizer, etc.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this notebook's PyTorch environment")

device = torch.device("cuda")
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
print(torch.cuda.get_device_name(0), dtype)

model = TextGenerationModel(**model_config).to(device=device, dtype=dtype)

NVIDIA GeForce RTX 4070 torch.bfloat16


The number of transformer blocks and the hidden dimension directly determine the model size. You can think of them as the “depth” and “width” of the model respectively. For each transformer block, you need to specify the number of attention heads (and in GQA, the number of key-value heads). Since we’re using the MoE model, you also need to define the total number of experts and the top-k value. Note that the MLP sublayer (implemented as SwiGLU) typically sets the intermediate dimension to 4 times the hidden dimension, so you don’t need to specify this separately.

The remaining hyperparameters don’t affect the model size: the maximum sequence length (which the rotary positional encoding depends on), the vocabulary size (which determines the embedding matrix dimensions), and the dropout rate used during training.

In this model, we create a single rotary position encoding module that’s reused across all transformer blocks. Since it’s a constant module, we only need one instance. The model begins with an embedding layer that converts token IDs into embedding vectors. These vectors are then processed through a series of transformer blocks. The output from the final transformer block remains a sequence of embedding vectors, which we normalize and project to vocabulary-sized logits using a linear layer. These logits represent probability distributions for predicting the next token in the sequence.

## Question: Why Does the Transformer Model Accept an Optional Attention Mask?

The `TextGenerationModel.forward()` method accepts `mask` as an optional argument:

```python
def forward(self, ids, mask=None):
    x = self.embedding(ids)
    for decoder in self.decoders:
        x = decoder(x, mask, self.rope)
    x = self.norm(x)
    return self.out(x)
```

Since decoder-only Transformers use causal attention, wouldn't it make more sense to generate the causal mask internally?

## Answer

**Yes. A decoder-only Transformer should enforce causal masking internally by default.** However, accepting an optional external mask provides flexibility for other attention constraints, such as padding masks and custom attention patterns.

### 1. Causal Masking Is a Model Requirement

In autoregressive language models, token $i$ must not attend to future tokens $j > i$.

The causal mask is defined as:

$$
M_{ij}^{\text{causal}} =
\begin{cases}
0, & j \leq i \\
-\infty, & j > i
\end{cases}
$$

where:

- $i$: Query token position
- $j$: Key token position
- $0$: Attention is permitted
- $-\infty$: Attention is blocked

For a sequence of length $N=4$:

$$
M^{\text{causal}} =
\begin{bmatrix}
0 & -\infty & -\infty & -\infty \\
0 & 0 & -\infty & -\infty \\
0 & 0 & 0 & -\infty \\
0 & 0 & 0 & 0
\end{bmatrix}
$$

The attention equation becomes:

$$
\operatorname{Attention}(Q,K,V)
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}} + M
\right)V
$$

Therefore, causal masking must always be enforced during standard autoregressive language modeling.

### 2. Why Accept an External Mask?

An optional `mask` argument supports additional requirements:

| Mask Type | Purpose |
|---|---|
| Causal mask | Prevent attention to future tokens |
| Padding mask | Prevent attention to padding tokens |
| Combined mask | Apply causal and padding constraints together |
| Custom attention mask | Support specialized attention patterns |

For example, suppose a sequence contains:

```text
Tokens:  [A, B, C, PAD, PAD]
```

The model should prevent attention to future tokens **and** prevent valid queries from attending to padding positions.

An external mask allows the caller to supply information about padding or other restrictions that may not be available within the attention layer.

### 3. Recommended Design: Internal Causal Mask + Optional External Mask

A practical design is to construct the causal mask internally and combine it with an optional external mask.

For a simple full-sequence self-attention implementation:

```python
def forward(self, ids, mask=None):
    B, N = ids.shape

    # True means attention is allowed
    causal_mask = torch.ones(
        N, N, device=ids.device, dtype=torch.bool
    ).tril()

    # Combine with optional external Boolean mask
    if mask is not None:
        # External mask must use True = allowed
        attention_mask = causal_mask & mask
    else:
        attention_mask = causal_mask

    x = self.embedding(ids)

    for decoder in self.decoders:
        x = decoder(x, attention_mask, self.rope)

    x = self.norm(x)
    return self.out(x)
```

**Important implementation details:**

- This example assumes that `GQA` accepts a Boolean mask with `True` meaning attention is permitted.
- For batched multi-head attention, the mask must be broadcastable to the attention score shape $(B,H,N,N)$. A batch-specific mask of shape $(B,N,N)$ can be converted to $(B,1,N,N)$ using `unsqueeze(1)`.
- Fully masked query rows require careful handling to avoid numerical problems.
- During autoregressive generation with a key-value (KV) cache, the query and key lengths may differ, so causal masking must account for cached token positions.
- If the attention implementation already enforces causality using `is_causal=True`, an additional explicit causal mask is generally unnecessary.

### 4. Advantages of This Design

1. **Correctness:** Causal masking is enforced even when the caller does not provide a mask.
2. **Flexibility:** External masks can specify padding or custom attention restrictions.
3. **Convenience:** Users do not need to construct a causal mask for every forward pass.
4. **Maintainability:** Causal masking remains part of the model's internal attention logic.
5. **Efficiency:** Implementations may use optimized causal attention kernels instead of explicitly constructing a full mask.

---

## Key Takeaway

The `mask=None` argument is useful for flexibility, but it should **not mean that causal masking is disabled**.

For a standard decoder-only language model, the recommended behavior is:

$$
\boxed{
M_{\text{effective}}
=
M_{\text{causal}}
\cap
M_{\text{external}}
}
$$

where the intersection represents the logical AND of allowed attention positions. If no external mask is provided, only the causal mask is applied.

**Conclusion:** Generate or enforce the causal mask internally by default, while allowing an optional external mask for padding and other attention constraints. This keeps the model autoregressively correct and flexible.


# Lesson 9: Training the Model

In lesson 1, We prepared the dataset for training. The next step is to wrap the dataset as a PyTorch Dataset object:

In [19]:
'''
Create PyTorch dataset

Method 1:the dataset advances by just one token between examples. 
That creates highly overlapping training samples and 44,943 batches per epoch.
''' 
# class GutenbergDataset(torch.utils.data.Dataset):
#     def __init__(self, text, tokenizer, seq_len=512):
#         self.seq_len = seq_len
#         # Encode the entire text
#         self.encoded = tokenizer.encode(text).ids

#     def __len__(self):
#         return len(self.encoded) - self.seq_len

#     def __getitem__(self, idx):
#         chunk = self.encoded[idx:idx + self.seq_len + 1]  # +1 for target
#         x = torch.tensor(chunk[:-1])
#         y = torch.tensor(chunk[1:])
#         return x, y

'''
Method 2: At batch size 32, the batch count should fall from 44,943 to roughly 88 per epoch.
This changes the training data sampling: each epoch covers the text once with nonoverlapping 
sequences, rather than repeatedly training on nearly identical windows.
'''
# class GutenbergDataset(torch.utils.data.Dataset):
#     def __init__(self, text, tokenizer, seq_len=512):
#         self.seq_len = seq_len
#         self.encoded = tokenizer.encode(text).ids

#     def __len__(self):
#         return max(0, (len(self.encoded) - 1) // self.seq_len)

#     def __getitem__(self, idx):
#         start = idx * self.seq_len
#         chunk = self.encoded[start:start + self.seq_len + 1]
#         return torch.tensor(chunk[:-1]), torch.tensor(chunk[1:])

'''
With a one-token stride, the model sees many nearly identical 
512-token windows; with a 512-token stride, it sees each passage
once per epoch. Training becomes much faster, but the model gets
fewer parameter updates and may perform differently.

Method 3: A good compromise is a stride of 128 or 256 tokens. 
It retains some overlap while cutting the batch count by roughly 128× or 256×:
'''
class GutenbergDataset(torch.utils.data.Dataset):
    def __init__(self, text, tokenizer, seq_len=512, stride=128):
        self.seq_len = seq_len
        self.stride = stride
        self.encoded = tokenizer.encode(text).ids

    def __len__(self):
        return max(0, (len(self.encoded) - self.seq_len - 1) // self.stride + 1)
    
    def __getitem__(self, idx):
        start = idx * self.stride
        chunk = self.encoded[start:start + self.seq_len + 1]
        return torch.tensor(chunk[:-1]), torch.tensor(chunk[1:])

##  Stride: Controlling Training Window Overlap

The parameter `stride` controls where each training example starts in the tokenized dataset.

Each training example contains:

- `seq_len` input tokens.
- One additional token to construct the shifted target sequence.

For example, when:

```python
seq_len = 512
stride = 128
```

The starting positions are:

$$
0,\ 128,\ 256,\ 384,\ 512,\ \ldots
$$

The overlap between consecutive input windows is:

$$
\text{Overlap} = \max(0,\text{seq\_len} - \text{stride})
$$

For `seq_len = 512` and `stride = 128`:

$$
\text{Overlap} = 512 - 128 = \boxed{384\text{ tokens}}
$$

### Stride Comparison

| Stride | Overlap Between 512-Token Examples | Effect |
|---:|---:|---|
| 1 | 511 tokens | Many nearly identical examples; high computational cost |
| 128 | 384 tokens | More training views at moderate computational cost |
| 512 | 0 tokens | Fewer training examples and updates per epoch |

### Example: Creating Training Windows

```python
start = idx * stride

# Input sequence
x = tokens[start : start + seq_len]

# Next-token prediction targets
y = tokens[start + 1 : start + seq_len + 1]
```

Here:

- `x` contains 512 input tokens.
- `y` contains 512 target tokens, shifted one position forward.
- The dataset needs 513 consecutive tokens to construct both sequences.

**Important:** Stride changes how the training text is sampled. It does **not** change the model's context length.

A smaller stride produces more overlapping training examples, while a larger stride reduces overlap and the number of training windows.


This dataset is designed for model pre-training, where the task is to predict the next token in a sequence. The dataset object is a Python iterable that produces pairs of (x,y), where x is a sequence of token IDs with fixed length, and y is the corresponding next token. Since the training targets (y) are derived from the input data itself, this approach is called self-supervised learning.

Depending on your hardware, you can optimize the training speed and memory usage. If you have a GPU with limited memory, you can load the model onto the GPU and use half-precision (bfloat16) to reduce memory consumption. Here’s how:  

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  
model = model.to(device).to(torch.bfloat16)  

If you still encounter out of memory error, you may want to reduce the model size or batch size.



In [20]:
# Create dataset and dataloader
BATCH_SIZE = 32
text = "\n".join(get_dataset_text())
dataset = GutenbergDataset(text, tokenizer, seq_len=model_config["max_seq_len"])
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

You need to write a training loop to train the model. In PyTorch, you may do as follows:

In [21]:
# Training loop
if os.path.exists("textgen_model.pth"):
    model.load_state_dict(torch.load("textgen_model.pth"))
else:
    N_EPOCHS = 5
    LR = 2e-4
    total_steps = N_EPOCHS * len(dataloader)
    WARMUP_STEPS = max(1, int(0.05 * total_steps))
    CLIP_NORM = 6.0

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.token_to_id("[pad]"))

    # Learning rate scheduling
    warmup_scheduler = optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_STEPS)
    cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=N_EPOCHS * len(dataloader) - WARMUP_STEPS, eta_min=0)
    scheduler = optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[WARMUP_STEPS])

    print(f"Training for {N_EPOCHS} epochs with {len(dataloader)} steps per epoch")
    best_loss = float('inf')

    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0

        progress_bar = tqdm.tqdm(dataloader, desc=f"Epoch {epoch+1}/{N_EPOCHS}")
        for x, y in progress_bar:
            x = x.to(device)
            y = y.to(device)

            # Create causal mask
            mask = create_causal_mask(x.shape[1], device, dtype)

            # Forward pass
            optimizer.zero_grad()
            outputs = model(x, mask.unsqueeze(0))

            # Compute loss
#             loss = loss_fn(outputs.view(-1, outputs.shape[-1]), y.view(-1))
            loss = loss_fn(outputs.float().reshape(-1, outputs.shape[-1]), y.reshape(-1))

            # Backward pass
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), CLIP_NORM, error_if_nonfinite=True
            )
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()

            # Show loss in tqdm
            progress_bar.set_postfix(loss=loss.item())

        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{N_EPOCHS}; Avg loss: {avg_loss:.4f}")

        # Save checkpoint if loss improved
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), "textgen_model.pth")

Training for 5 epochs with 352 steps per epoch


Epoch 1/5: 100%|██████████| 352/352 [59:40<00:00, 10.17s/it, loss=4.84] 


Epoch 1/5; Avg loss: 5.6264


Epoch 2/5: 100%|██████████| 352/352 [59:01<00:00, 10.06s/it, loss=3.81]


Epoch 2/5; Avg loss: 4.0416


Epoch 3/5: 100%|██████████| 352/352 [57:41<00:00,  9.83s/it, loss=2.78]


Epoch 3/5; Avg loss: 3.3252


Epoch 4/5: 100%|██████████| 352/352 [57:59<00:00,  9.88s/it, loss=2.82]


Epoch 4/5; Avg loss: 2.7368


Epoch 5/5: 100%|██████████| 352/352 [57:56<00:00,  9.88s/it, loss=2.57]


Epoch 5/5; Avg loss: 2.5672


While this training loop might differ from what you’ve used for other models, it follows best practices for training transformers. The code uses a cosine learning rate scheduler with a warm-up period—the learning rate gradually increases during warm-up and then decreases following a cosine curve.

To prevent gradient explosion, we implement gradient clipping, which stabilizes training by limiting drastic changes in model parameters.

The model functions as a next-token predictor, outputting a probability distribution over the entire vocabulary. Since this is essentially a classification task (predicting which token comes next), we use cross-entropy loss for training.

The training progress is monitored using tqdm, which displays the loss for each epoch. The model’s parameters are saved whenever the loss improves, ensuring we keep the best performing version.

# Lesson 10: Using the Model

After training the model, you can use it to generate text. To optimize performance, disable gradient computation in PyTorch. Additionally, since some modules like dropout behave differently during training and inference, switch the model to evaluation mode before use.

Let’s create a function for text generation that can be called multiple times to generate different samples:

In [22]:
# Generation function
def generate_text(model, tokenizer, prompt, max_length=100, temperature=0.7):
    model.eval()
    device = next(model.parameters()).device

    # Encode the prompt
    input_ids = torch.tensor(tokenizer.encode(prompt).ids).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_length):
            # Get model predictions for the next token as the last element of the output
            context = input_ids[:, -model_config["max_seq_len"]:]
            mask = create_causal_mask(context.shape[1], device, dtype)
            outputs = model(context, mask.unsqueeze(0))

#             outputs = model(input_ids)
            next_token_logits = outputs[:, -1, :] / temperature
            # Sample from the distribution
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            # Append to input_ids
            input_ids = torch.cat([input_ids, next_token], dim=1)
            # Stop if we predict the end token
            if next_token[0].item() == tokenizer.token_to_id("[eos]"):
                break

    return tokenizer.decode(input_ids[0].tolist())

# Test the model with some prompts
test_prompts = [
    "Once upon a time,",
    "We the people of the",
    "In the beginning was the",
]

print("\nGenerating sample texts:")
for prompt in test_prompts:
    generated = generate_text(model, tokenizer, prompt)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 80)


Generating sample texts:

Prompt: Once upon a time,
Generated:  Once upon a time, up from the North; for I saw the most citizen of
that ground around. There I must get through, and be able to go down and,
and then put into the paths at once. Begone on the road to destroy the
whole country, so that I would not remain in my opponents; but the
condition of the cases I had at West Point out, and that I would not
attack us.  When I felt that I might say this, and I wanted
--------------------------------------------------------------------------------

Prompt: We the people of the
Generated:  We the people of the various people who were all their in
the property, would be, indeed, be unconsciously worship.  In this
generals, if a man should be absurd to go without some conditions
of life, it would be necessary to hold a line where the country
would be so far as to do it.  It would there be immediate objection to
the contractor of the other, or at the name of many a large number of
governme

The generate_text() function implements probabilistic sampling for token generation. Although the model outputs logits representing a probability distribution over the vocabulary, it doesn’t always select the most probable token. Instead, it uses the softmax function to convert logits to probabilities. The temperature parameter controls the sampling distribution: lower values make the model more conservative by emphasizing likely tokens, while higher values make it more creative by reducing the probability differences between tokens.

The function takes a partial sentence as a prompt string and generates a sequence of tokens using the model. Although the model is trained with batches, this function uses a batch size of 1 for simplicity. The final output is returned as a decoded string.

## Temperature: Controlling Randomness During Text Generation

The parameter `temperature` controls the randomness of next-token sampling during text generation.

The model produces logits for candidate tokens. Before applying softmax, the logits are divided by temperature:

$$
\boxed{
P(x_i)=
\frac{\exp(z_i/T)}
{\sum_{j=1}^{V}\exp(z_j/T)}
}
$$

where:

- $P(x_i)$: Probability of selecting token $i$.
- $z_i$: Logit assigned to token $i$ by the model.
- $T$: Sampling temperature.
- $V$: Vocabulary size.

### PyTorch Implementation

```python
import torch.nn.functional as F

probs = F.softmax(
    next_token_logits / temperature,
    dim=-1
)
```

Temperature must be greater than zero:

$$
T > 0
$$

### Temperature Comparison

| Temperature | Effect | Typical Behavior |
|---|---|---|
| $T < 1$ | Sharpens the probability distribution | More predictable output |
| $T = 1$ | Preserves the original softmax distribution | Normal sampling |
| $T > 1$ | Flattens the probability distribution | More varied output |

### Examples

**Low temperature: $T = 0.7$**

- Favors high-probability tokens more strongly.
- Reduces randomness.
- Often produces more predictable text.

**Default temperature: $T = 1.0$**

- Uses the model's original softmax probability distribution.
- Does not modify the relative logits before softmax.

**High temperature: $T = 1.2$**

- Gives lower-probability tokens a greater chance of selection.
- Increases output diversity.
- May reduce coherence when the temperature is too high.

### Important Distinction

Temperature affects **sampling during text generation**, not model training.

Changing temperature does not update the model's parameters.

For the current model, try:

```python
temperature = 0.7
```

Then compare several generated outputs from the same prompt using different temperatures.

---

## 3. Stride vs. Temperature

| Property | Stride | Temperature |
|---|---|---|
| Stage | Training data preparation | Text generation |
| Main purpose | Control training window overlap | Control sampling randomness |
| Affects model weights? | Indirectly, through training data | No |
| Changes context length? | No | No |
| Smaller value | More overlapping examples | More concentrated probabilities |
| Larger value | Fewer overlapping examples | Flatter probability distribution |

### Key Takeaways

1. **Stride** determines how far the training window moves between consecutive examples.
2. Smaller strides generate more overlapping examples and increase computational cost.
3. **Temperature** changes the probability distribution used to sample the next token.
4. Lower temperatures generally produce more predictable text; higher temperatures produce more diverse text.
5. Stride is a **training data parameter**, while temperature is a **generation-time parameter**.

$$
\boxed{
\text{Stride: Training Data Sampling}
\qquad
\text{Temperature: Token Sampling}
}
$$